# illust-asset-forge

上から順に実行する。1 つ目のコードセルはモデルを読み込むので数分かかる。

## 1. 準備

Drive をつなぎ、リポジトリを取り、ライブラリを入れてモデルを読み込む。

In [ ]:
import os
import subprocess
from google.colab import drive

drive.mount("/content/drive")

REPO = "/content/illust-asset-forge"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "https://github.com/akabee0161/illust-asset-forge.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull"], check=True)

!pip install -q -r {REPO}/requirements-colab.txt
!pip install -q -e {REPO} --no-deps

os.chdir(REPO)
from artgen import forge as F

forge = F.load(".")

## 2. 顔（通常）

8 枚作って 1 枚選ぶ。選んだファイル名は次のセルで使う。

In [ ]:
paths = forge.generate("ally05", "face", n=8)
F.show(paths)

## 3. 表情

採用した顔を `source` に渡す。`inpaint_box` が要るので、顔を採用して Claude が書いた後に実行する。表情名は `character.yaml` の `expressions` のキー。

In [ ]:
paths = forge.inpaint("ally05", "hurt", source="/content/drive/MyDrive/illust-asset-forge/runs/<run_id>/<採用した顔>.png", n=4)
F.show(paths)

## 4. 全身（参考資料）

In [ ]:
paths = forge.generate("ally05", "fullbody", n=4)
F.show(paths)

## 試すときの書き方

- 足すタグ・消すタグ: `forge.generate("ally05", "face", extra="scar on cheek", remove="stubble")`（ネガティブは `extra_negative=`・`remove_negative=`）
- シードを固定: `seeds=[123, 456]`（渡すと `n` は無視される）
- guidance・steps: `guidance=6`、`steps=30`
- **YAML は Colab で書き換えない**（次の pull でぶつかる）。採用した PNG を zip にして Claude に渡す。上書きの内容は PNG の条件（`overrides`）に残るので、Claude が設定に戻す。